## Pull the data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo

In [2]:
mi = fetch_ucirepo(id=579)

print("Dataset fetched successfully!")
print("Dataset name:", mi.metadata.name)
print("Number of instances:", mi.metadata.num_instances)



Dataset fetched successfully!
Dataset name: Myocardial infarction complications
Number of instances: 1700


In [3]:
X_raw = mi.data.features.copy()
y = mi.data.targets.copy()

print("Input rows,columns:", X_raw.shape)
print("Output rows,columns:", y.shape)

Input rows,columns: (1700, 111)
Output rows,columns: (1700, 12)


## Drop Past 48 hours data - columns 95(R_AB_3_n), 102(NA_R_3_n), and 105(NOT_NA_3_n) --> ID column already dropped when grabbing input set

In [4]:
drop_columns = [
    "R_AB_3_n",
    "NA_R_3_n",
    "NOT_NA_3_n"
]

X_48h = X_raw.drop(columns=drop_columns)

print("New input rows,columns:", X_48h.shape)
print("Output rows,columns:", y.shape)

New input rows,columns: (1700, 108)
Output rows,columns: (1700, 12)


## Now check for data missing values and data types - Data Analysis

In [5]:
#define missing
missing_vals = X_48h.isnull().sum()

#define missing percentage
missing_percent = (missing_vals / len(X_48h)) * 100

#put in dataframe
missing_table_df = pd.DataFrame({
    "Missing Values Count": missing_vals,
    "Missing Values Percent": missing_percent
})

# Display columns with missing values sorted in order of highest eprcentage of missing values
missing_table_values = missing_table_df[
    missing_table_df["Missing Values Count"] > 0
]

missing_table = missing_table_values.sort_values(
    by="Missing Values Percent",
    ascending=False
)

print(missing_table)

            Missing Values Count  Missing Values Percent
KFK_BLOOD                   1696               99.764706
IBS_NASL                    1628               95.764706
D_AD_KBRIG                  1076               63.294118
S_AD_KBRIG                  1076               63.294118
NOT_NA_KB                    686               40.352941
...                          ...                     ...
zab_leg_06                     7                0.411765
zab_leg_04                     7                0.411765
NA_R_1_n                       5                0.294118
INF_ANAM                       4                0.235294
IM_PG_P                        1                0.058824

[107 rows x 2 columns]


In [6]:
print(missing_table.to_string())

               Missing Values Count  Missing Values Percent
KFK_BLOOD                      1696               99.764706
IBS_NASL                       1628               95.764706
D_AD_KBRIG                     1076               63.294118
S_AD_KBRIG                     1076               63.294118
NOT_NA_KB                       686               40.352941
LID_KB                          677               39.823529
NA_KB                           657               38.647059
GIPER_NA                        375               22.058824
NA_BLOOD                        375               22.058824
K_BLOOD                         371               21.823529
GIPO_K                          369               21.705882
AST_BLOOD                       285               16.764706
ALT_BLOOD                       284               16.705882
D_AD_ORIT                       267               15.705882
S_AD_ORIT                       267               15.705882
DLIT_AG                         248     

## The first 17 features have over 10% data missing ranging from over 11% to over 99%. Investigate column information for those

In [7]:
highest_missing_value_features = missing_table.head(17).index

mi_variables_filtered = mi.variables[
    mi.variables["name"].isin(highest_missing_value_features)
]

print(mi_variables_filtered.to_string(index=False))

      name    role        type demographic                                                                                                                                                     description  units missing_values
  IBS_NASL Feature      Binary         NaN                                                                                                             Heredity on CHD\n\n0: isn't burdened\n\n1: burdened    NaN            yes
   DLIT_AG Feature Categorical         NaN there was no arterial hypertension\n\n1: one year\n\n2: two years\n\n3: three years\n\n4: four years\n\n5: five years\n\n6: 6-10 years\n\n7: more than 10 years    NaN            yes
S_AD_KBRIG Feature     Integer         NaN                                                                                                 Systolic blood pressure according to Emergency Cardiology Team    mmHg            yes
D_AD_KBRIG Feature     Integer         NaN                                                          

## Dropping Columns KFK_BLOOD and IBS_NASL for sure as they have above 95% values missing (KFK_BLOOD has only 4 values and IBS_NASL has 72)

In [8]:
X_drop_1 = X_48h.drop(columns=["KFK_BLOOD", "IBS_NASL"])

In [9]:
print("Missing Value Feature Drop input shape:", X_drop_1.shape)
print("KFK_BLOOD still in set:", "KFK_BLOOD" in X_drop_1.columns)
print("IBS_NASL still int set:", "IBS_NASL" in X_drop_1.columns)

Missing Value Feature Drop input shape: (1700, 106)
KFK_BLOOD still in set: False
IBS_NASL still int set: False


## Now investigating some other high percentage missing data -  (D_AD_KBRIG - 63.294118) and (S_AD_KBRIG - 63.294118)



These have above 50% of the data missing which is a massive issue. These columns are described as Systolic blood pressure according to Emergency Cardiology Team and Diastolic blood pressure according to Emergency Cardiology Team. It can be assumed that a high amount of missing values - not everyone was brought in in an emergency vehicle - some people might have checked in or been driven directly to the hospital bypassing those readings. Furthermore we have coolumns: S_AD_ORIT (Systolic blood pressure according to intensive care unit) and D_AD_ORIT (Diastolic blood pressure according to intensive care unit) which only have aorunf 15% of data missing and are readings taken in ICU(more updated). Since we are predicting on information available with 48 Hours - this data would be avaible to us and might be more updated. Running a correlation analysis:

In [10]:
blood_pressure_pairs = [("D_AD_KBRIG", "D_AD_ORIT"), ("S_AD_KBRIG", "S_AD_ORIT")]

for col1, col2 in blood_pressure_pairs:

    paired_data = X_drop_1[[col1, col2]]

    paired_data = paired_data.dropna()

    print(f"\n{col1} vs {col2}")

    print("Paired observations:", len(paired_data))

    correlation = paired_data[col1].corr(paired_data[col2])

    print("Pearson correlation:", correlation)


D_AD_KBRIG vs D_AD_ORIT
Paired observations: 585
Pearson correlation: 0.5559600310828933

S_AD_KBRIG vs S_AD_ORIT
Paired observations: 585
Pearson correlation: 0.6113647279368555


There are only 585 instances where both readings exist (the KBRIG and ORIT). The correlaation score for them is moderate however the high level of missingness is still an issue. We can keep the D_AD_KBRIG and S_AD_KBRIG columns, impute them and then create a binary column EMR_RES_BLOOD with 1 if data exists or 0 if it doesn't. First we need to check if every instance with a missing value for D_AD_KBRIG and S_AD_KBRIG aline.

In [11]:
diastolic_missing = X_drop_1["D_AD_KBRIG"].isna()
systolic_missing = X_drop_1["S_AD_KBRIG"].isna()

same_missingness = diastolic_missing.equals(
    systolic_missing
)

print(
    "Same missingness pattern:",
    same_missingness
)

Same missingness pattern: True


This means everytime one measurement is missing, so is the other. So we only need one binary indicator column.

## Create Binary Indicator Column

In [12]:
X_indicator = X_drop_1.copy()

data_exists_bp = X_indicator["D_AD_KBRIG"].notna()

emr_res_indicator = data_exists_bp.astype(int)

X_indicator["EMR_RES_BLOOD"] = emr_res_indicator

print("BP measurements available:", X_indicator["EMR_RES_BLOOD"].sum())
print("BP measurements missing:", (X_indicator["EMR_RES_BLOOD"] == 0).sum())

print("OG 48-hour features:", X_48h.shape)
print("After removing high missing KFK_BLOOD and IBS_NASL:", X_drop_1.shape)
print("After adding Binary indicator for emergency response:", X_indicator.shape)


BP measurements available: 624
BP measurements missing: 1076
OG 48-hour features: (1700, 108)
After removing high missing KFK_BLOOD and IBS_NASL: (1700, 106)
After adding Binary indicator for emergency response: (1700, 107)


## RUN FEATURE TO FEATURE CORRELATION ANALYSIS NOW

Correlation analysis works on continous features only so we grab all continous features. Note that there might be integer values in data as well however it needs to be seprated with integer data that represents categorical encoding so we will work with only continous values for now

In [ ]:
continuous_column_names_from_metadata = mi.variables.loc[mi.variables["type"] == "Continuous", "name"].tolist()

# Grab ones only if we decided to keep that column
continuous_column_names_keep = []

for column_name in continuous_column_names_from_metadata:
    if column_name in X_indicator.columns:
        continuous_column_names_keep.append(column_name)



print("Continuous features in metadata:", len(continuous_column_names_from_metadata))
print(continuous_column_names_from_metadata)
print("Continuous features kept:", len(continuous_column_names_keep))
print(continuous_column_names_keep)

Continuous features in metadata: 7
['K_BLOOD', 'NA_BLOOD', 'ALT_BLOOD', 'AST_BLOOD', 'KFK_BLOOD', 'L_BLOOD', 'ROE']
Continuous features kept: 6
['K_BLOOD', 'NA_BLOOD', 'ALT_BLOOD', 'AST_BLOOD', 'L_BLOOD', 'ROE']


In [14]:
# Calculate correlation of the kept continous features
continuous_correlation_matrix = X_indicator[continuous_column_names_keep].corr(method="pearson")

display(continuous_correlation_matrix.round(2))

,K_BLOOD,NA_BLOOD,ALT_BLOOD,AST_BLOOD,L_BLOOD,ROE
K_BLOOD,1.00,0.30,0.02,0.05,0.01,0.01
NA_BLOOD,0.30,1.00,-0.01,-0.02,0.02,-0.01
ALT_BLOOD,0.02,-0.01,1.00,0.52,0.04,-0.01
AST_BLOOD,0.05,-0.02,0.52,1.00,0.08,-0.03
L_BLOOD,0.01,0.02,0.04,0.08,1.00,0.01
ROE,0.01,-0.01,-0.01,-0.03,0.01,1.00


Looking at the correlation - it does not seem like anything is highly correlated enough to be dropped based on this alone. None of the features seem highly redundant.

Now we can check the integer values and their descriptions to separate actual integer values vs categorical encodings

In [ ]:
integer_column_names_from_metadata = mi.variables.loc[mi.variables["type"] == "Integer", "name"].tolist()


integer_column_names_keep = []

for column_name in integer_column_names_from_metadata:
    if column_name in X_indicator.columns:
        integer_column_names_keep.append(column_name)


integer_metadata_details = mi.variables

integer_metadata_details_filter = integer_metadata_details[integer_metadata_details["name"].isin(integer_column_names_keep)]

integer_metadata_details_keep = integer_metadata_details_filter[["name", "type", "description", "units"]].copy()

print("Integer features in metadata:", len(integer_column_names_from_metadata))
print(integer_column_names_from_metadata)
print("Integer features kept:", len(integer_column_names_keep))
display(integer_metadata_details_keep)

Integer features in metadata: 11
['ID', 'AGE', 'S_AD_KBRIG', 'D_AD_KBRIG', 'S_AD_ORIT', 'D_AD_ORIT', 'NA_R_1_n', 'NA_R_2_n', 'NA_R_3_n', 'NOT_NA_2_n', 'NOT_NA_3_n']
Integer features kept: 8


,name,type,description,units
1,AGE,Integer,Age of patient.,NaN
34,S_AD_KBRIG,Integer,Systolic blood pressure according to Emergency...,mmHg
35,D_AD_KBRIG,Integer,Diastolic blood pressure according to Emergenc...,mmHg
36,S_AD_ORIT,Integer,Systolic blood pressure according to intensive...,mmHg
37,D_AD_ORIT,Integer,Diastolic blood pressure according to intensiv...,mmHg
99,NA_R_1_n,Integer,Use of opioid drugs in the ICU in the first ho...,NaN
100,NA_R_2_n,Integer,Use of opioid drugs in the ICU in the second d...,NaN
103,NOT_NA_2_n,Integer,Use of NSAIDs in the ICU in the second day of ...,NaN


Based on those results along with the descriptions on the dataset provided, only AGE, S_AD_ORIT, D_AD_ORIT,S_AD_KBRIG and D_AD_KBRIG are actually integer values that don't represent categorical encoding.

In [45]:
integer_measurement_column_names_keep = [
    "AGE",
    "S_AD_ORIT",
    "D_AD_ORIT",
    "S_AD_KBRIG",
    "D_AD_KBRIG"
]

numerical_measurement_column_names_keep = (
    continuous_column_names_keep
    + integer_measurement_column_names_keep
)

print("Total numerical measurement features:", len(numerical_measurement_column_names_keep))
print(numerical_measurement_column_names_keep)

Total numerical measurement features: 11
['K_BLOOD', 'NA_BLOOD', 'ALT_BLOOD', 'AST_BLOOD', 'L_BLOOD', 'ROE', 'AGE', 'S_AD_ORIT', 'D_AD_ORIT', 'S_AD_KBRIG', 'D_AD_KBRIG']


In [46]:
numerical_measurement_correlation_matrix = X_indicator[numerical_measurement_column_names_keep].corr(method="pearson")

display(numerical_measurement_correlation_matrix.round(2))

,K_BLOOD,NA_BLOOD,ALT_BLOOD,AST_BLOOD,L_BLOOD,ROE,AGE,S_AD_ORIT,D_AD_ORIT,S_AD_KBRIG,D_AD_KBRIG
K_BLOOD,1.00,0.30,0.02,0.05,0.01,0.01,-0.00,0.03,0.01,-0.00,-0.01
NA_BLOOD,0.30,1.00,-0.01,-0.02,0.02,-0.01,0.03,0.04,0.02,-0.01,0.01
ALT_BLOOD,0.02,-0.01,1.00,0.52,0.04,-0.01,-0.10,-0.10,-0.06,-0.05,-0.06
AST_BLOOD,0.05,-0.02,0.52,1.00,0.08,-0.03,-0.05,-0.10,-0.08,-0.08,-0.06
L_BLOOD,0.01,0.02,0.04,0.08,1.00,0.01,0.00,-0.14,-0.16,-0.17,-0.13
ROE,0.01,-0.01,-0.01,-0.03,0.01,1.00,0.21,0.04,0.01,0.01,-0.02
AGE,-0.00,0.03,-0.10,-0.05,0.00,0.21,1.00,0.04,-0.05,0.10,-0.02
S_AD_ORIT,0.03,0.04,-0.10,-0.10,-0.14,0.04,0.04,1.00,0.86,0.61,0.54
D_AD_ORIT,0.01,0.02,-0.06,-0.08,-0.16,0.01,-0.05,0.86,1.00,0.56,0.56
S_AD_KBRIG,-0.00,-0.01,-0.05,-0.08,-0.17,0.01,0.10,0.61,0.56,1.00,0.84


Once again running correlation analysis - we can see that S_AD_ORIT and D_AD_ORIT have a strong positive correlation: 0.86. They provide similar information, however this is to be expected as they are different measurements of blood pressure. Similarily, S_AD_KBRIG and D_AD_BBRIG have the same situation witha. strong positive correlation of 0.84. Although there is a stribg correlation, we will keep all features as they are different aspects of blood pressure measurement.

## Figure out classifications of other columns - remaining data types

Count of the features not analyzed yet(non continous or integer)

In [55]:
other_feature_column_names_keep = []

for column_name in X_indicator.columns:
    if column_name not in numerical_measurement_column_names_keep:
        other_feature_column_names_keep.append(column_name)

print("Features left:", len(other_feature_column_names_keep))
print(other_feature_column_names_keep)

Features left: 96
['SEX', 'INF_ANAM', 'STENOK_AN', 'FK_STENOK', 'IBS_POST', 'GB', 'SIM_GIPERT', 'DLIT_AG', 'ZSN_A', 'nr_11', 'nr_01', 'nr_02', 'nr_03', 'nr_04', 'nr_07', 'nr_08', 'np_01', 'np_04', 'np_05', 'np_07', 'np_08', 'np_09', 'np_10', 'endocr_01', 'endocr_02', 'endocr_03', 'zab_leg_01', 'zab_leg_02', 'zab_leg_03', 'zab_leg_04', 'zab_leg_06', 'O_L_POST', 'K_SH_POST', 'MP_TP_POST', 'SVT_POST', 'GT_POST', 'FIB_G_POST', 'ant_im', 'lat_im', 'inf_im', 'post_im', 'IM_PG_P', 'ritm_ecg_p_01', 'ritm_ecg_p_02', 'ritm_ecg_p_04', 'ritm_ecg_p_06', 'ritm_ecg_p_07', 'ritm_ecg_p_08', 'n_r_ecg_p_01', 'n_r_ecg_p_02', 'n_r_ecg_p_03', 'n_r_ecg_p_04', 'n_r_ecg_p_05', 'n_r_ecg_p_06', 'n_r_ecg_p_08', 'n_r_ecg_p_09', 'n_r_ecg_p_10', 'n_p_ecg_p_01', 'n_p_ecg_p_03', 'n_p_ecg_p_04', 'n_p_ecg_p_05', 'n_p_ecg_p_06', 'n_p_ecg_p_07', 'n_p_ecg_p_08', 'n_p_ecg_p_09', 'n_p_ecg_p_10', 'n_p_ecg_p_11', 'n_p_ecg_p_12', 'fibr_ter_01', 'fibr_ter_02', 'fibr_ter_03', 'fibr_ter_05', 'fibr_ter_06', 'fibr_ter_07', 'fibr_ter

Display information regarding each column left

In [56]:
other_feature_metadata = mi.variables

other_feature_metadata_filter = other_feature_metadata[other_feature_metadata["name"].isin(other_feature_column_names_keep)]

other_feature_metadata_keep = other_feature_metadata_filter[["name", "type", "description", "units"]].copy()

display(other_feature_metadata_keep)

,name,type,description,units
2,SEX,Binary,"0: female, 1: male",NaN
3,INF_ANAM,Categorical,Quantity of myocardial infarctions in the anam...,NaN
4,STENOK_AN,Categorical,Exertional angina pectoris in the anamnesis. \...,NaN
5,FK_STENOK,Categorical,Functional class (FC) of angina pectoris in th...,NaN
6,IBS_POST,Categorical,"Coronary heart disease (CHD) in recent weeks, ...",NaN
...,...,...,...,...
107,ANT_CA_S_n,Binary,Use of calcium channel blockers in the ICU,NaN
108,GEPAR_S_n,Binary,Use of а anticoagulants (heparin) in the ICU,NaN
109,ASP_S_n,Binary,Use of acetylsalicylic acid in the ICU,NaN
110,TIKL_S_n,Binary,Use of Ticlid in the ICU,NaN


Find the split between each data type left # note as we are pulling from the tags left on their metadata we have to verify and also add EMR_RES_BLOOD as a binary feature

In [61]:
other_feature_metadata_type_counts = (other_feature_metadata_keep["type"].value_counts(dropna=False))

display(other_feature_metadata_type_counts)

type
Binary         77
Categorical    15
Integer         3
Name: count, dtype: int64

Analyze the ones that are defined as categorical first

In [62]:
categorical_column_names_keep = other_feature_metadata_keep.loc[other_feature_metadata_keep["type"] == "Categorical","name"].tolist()

categorical_metadata_details_keep = mi.variables.loc[mi.variables["name"].isin(categorical_column_names_keep),["name", "description"]].copy()

# Display each feature description 
for _, row in categorical_metadata_details_keep.iterrows():
    description = row["description"].replace("\\n", "\n")

    print(row["name"])

    print(description)

INF_ANAM
Quantity of myocardial infarctions in the anamnesis. 

0: zero

1: one

2: two

3: three and more
STENOK_AN
Exertional angina pectoris in the anamnesis. 

0: never

1: during the last year 

2: one year ago

3: two years ago

4: three years ago

5: 4-5 years ago
FK_STENOK
Functional class (FC) of angina pectoris in the last year. 

0: there is no angina pectoris

1: I FC

2: II FC

3: III FC

4: IV FC
IBS_POST
Coronary heart disease (CHD) in recent weeks, days before admission to hospital 

0: none

1: exertional angina pectoris

2: unstable angina pectoris
GB
Presence of an essential hypertension 

0: there is no essential hypertension 

1: Stage 1 

2: Stage 2

3: Stage 3
DLIT_AG
there was no arterial hypertension

1: one year

2: two years

3: three years

4: four years

5: five years

6: 6-10 years

7: more than 10 years
ZSN_A
Presence of chronic Heart failure (HF) in the anamnesis: Partially ordered attribute: there are two lines of severities:

0<1<2<4,

0<1<3<4.

State 

In [63]:
ordinal_column_names_keep = [
    "INF_ANAM", "STENOK_AN", "FK_STENOK", "GB", "DLIT_AG",
    "ant_im", "lat_im", "inf_im", "post_im", "TIME_B_S",
    "R_AB_1_n", "R_AB_2_n", "NOT_NA_1_n", "IBS_POST"
]

partially_ordered_column_names_keep = [
    "ZSN_A"
]

print("Ordinal categorical features:")
print(ordinal_column_names_keep)


print("\nPartially ordered categorical features:")
print(partially_ordered_column_names_keep)

print("\nTotal categorical features:",
      len(ordinal_column_names_keep)
      + len(partially_ordered_column_names_keep))

Ordinal categorical features:
['INF_ANAM', 'STENOK_AN', 'FK_STENOK', 'GB', 'DLIT_AG', 'ant_im', 'lat_im', 'inf_im', 'post_im', 'TIME_B_S', 'R_AB_1_n', 'R_AB_2_n', 'NOT_NA_1_n', 'IBS_POST']

Partially ordered categorical features:
['ZSN_A']

Total categorical features: 15


It can be seen that 14 of the 15 categorical inputs were defined as ordinal (meaning the categories associated with their numbers encoded, have some sort of ranking or order). We should preserve this meaning we leave them as during preprocessing as the categories prvide meaningful information via the order or ranking. 

The dataset has already one hot endocded nominal values(which is why there is a large number of binary columns) - values like ECG rhythms which was nominal was split into 3 columns and binary encoded. 

The Partially ordered categorical feature ZSN_A has 2 lines of severity:
0<1<2<4,
0<1<3<4.
State 4 means simultaneous states 2 and 3
							Cases	Fraction
0: there is no chronic heart failure			1468	86.35%
1: I stage						103	6.06%
2: IIÐ stage (heart failure due to right 
ventricular systolic dysfunction)			27	1.59%
3: IIÐ stage (heart failure due to left 
ventricular systolic dysfunction)			29	1.71%
4: IIB stage (heart failure due to left and
 right ventricular systolic dysfunction)		19	1.12%.

 This means we will be considering this column sperately and analyzing it to see if one hot encoding is needed.





VERIFY BINARY VALUES

In [ ]:
binary_column_names_keep = []

for column_name in X_indicator.columns:
    unique_values = set(X_indicator[column_name].dropna().unique())

    if unique_values.issubset({0, 1}):
        binary_column_names_keep.append(column_name)

print("Number of binary features:", len(binary_column_names_keep))

Number of binary features: 78


So all binary values tagged binary are correctly encoded with only 0 and 1

VERIFY THE 3 "INTEGER" VALUES

In [65]:
integer_metadata_details_from_other = other_feature_metadata_keep.loc[other_feature_metadata_keep["type"] == "Integer",["name", "type", "description", "units"]].copy()

display(integer_metadata_details_from_other)

,name,type,description,units
99,NA_R_1_n,Integer,Use of opioid drugs in the ICU in the first ho...,NaN
100,NA_R_2_n,Integer,Use of opioid drugs in the ICU in the second d...,NaN
103,NOT_NA_2_n,Integer,Use of NSAIDs in the ICU in the second day of ...,NaN


So the API call misclassfied it - it should be ordinal according to descrition and details given with dataset. Therefore let's add it to our ordinal list:

In [66]:
integer_ordinal_column_names_keep = []

integer_ordinal_column_names = (
    integer_metadata_details_from_other["name"].tolist()
)

for column_name in integer_ordinal_column_names:
    if column_name in X_indicator.columns:
        integer_ordinal_column_names_keep.append(column_name)

for column_name in integer_ordinal_column_names_keep:
    if column_name not in ordinal_column_names_keep:
        ordinal_column_names_keep.append(column_name)


In [67]:
print("X_indicator shape:", X_indicator.shape)

print("\nNumerical measurement features")
print("Count:", len(numerical_measurement_column_names_keep))
print(numerical_measurement_column_names_keep)

print("\nOrdinal features")
print("Count:", len(ordinal_column_names_keep))
print(ordinal_column_names_keep)

print("\nPartially Ordered")
print("Count:", len(partially_ordered_column_names_keep))
print(partially_ordered_column_names_keep)

print("\nBinary features")
print("Count:", len(binary_column_names_keep))
print(binary_column_names_keep)

all_classified_columns = (
    numerical_measurement_column_names_keep
    + ordinal_column_names_keep
    + partially_ordered_column_names_keep
    + binary_column_names_keep
)

print("\nComparison with X_indicator")
print("Unique classified features:", len(all_classified_columns))
print("Total X_indicator features:", X_indicator.shape[1])


X_indicator shape: (1700, 107)

Numerical measurement features
Count: 11
['K_BLOOD', 'NA_BLOOD', 'ALT_BLOOD', 'AST_BLOOD', 'L_BLOOD', 'ROE', 'AGE', 'S_AD_ORIT', 'D_AD_ORIT', 'S_AD_KBRIG', 'D_AD_KBRIG']

Ordinal features
Count: 17
['INF_ANAM', 'STENOK_AN', 'FK_STENOK', 'GB', 'DLIT_AG', 'ant_im', 'lat_im', 'inf_im', 'post_im', 'TIME_B_S', 'R_AB_1_n', 'R_AB_2_n', 'NOT_NA_1_n', 'IBS_POST', 'NA_R_1_n', 'NA_R_2_n', 'NOT_NA_2_n']

Partially Ordered
Count: 1
['ZSN_A']

Binary features
Count: 78
['SEX', 'SIM_GIPERT', 'nr_11', 'nr_01', 'nr_02', 'nr_03', 'nr_04', 'nr_07', 'nr_08', 'np_01', 'np_04', 'np_05', 'np_07', 'np_08', 'np_09', 'np_10', 'endocr_01', 'endocr_02', 'endocr_03', 'zab_leg_01', 'zab_leg_02', 'zab_leg_03', 'zab_leg_04', 'zab_leg_06', 'O_L_POST', 'K_SH_POST', 'MP_TP_POST', 'SVT_POST', 'GT_POST', 'FIB_G_POST', 'IM_PG_P', 'ritm_ecg_p_01', 'ritm_ecg_p_02', 'ritm_ecg_p_04', 'ritm_ecg_p_06', 'ritm_ecg_p_07', 'ritm_ecg_p_08', 'n_r_ecg_p_01', 'n_r_ecg_p_02', 'n_r_ecg_p_03', 'n_r_ecg_p_04'

## Analyze the feature set for missing values and imbalances to prep for test/validate/train splits

Before we split we have to figure out a way to split the dataset as class imbalance is a big issue - as some complications are rare, we have to find a way to split so each set gets positive cases. 

In [68]:
# Features(inputs)
X = X_indicator.copy()

# Targets(outputs)
y = mi.data.targets.copy()

print("Features shape:", X.shape)
print("Targets shape:", y.shape)

#Find if any target values have missing values
print("\nMissing target values:")
print(y.isnull().sum())

Features shape: (1700, 107)
Targets shape: (1700, 12)

Missing target values:
FIBR_PREDS    0
PREDS_TAH     0
JELUD_TAH     0
FIBR_JELUD    0
A_V_BLOK      0
OTEK_LANC     0
RAZRIV        0
DRESSLER      0
ZSN           0
REC_IM        0
P_IM_STEN     0
LET_IS        0
dtype: int64


In [ ]:
# Display let_is class distribution
let_is_class_distribution = y["LET_IS"].value_counts()

let_is_class_distribution = let_is_class_distribution.sort_index()

print("LET_IS class distribution:")
print(let_is_class_distribution)

# Calculate and display let_is class percentages
let_is_class_percentages = y["LET_IS"].value_counts(normalize=True)

let_is_class_percentages = let_is_class_percentages.sort_index()

let_is_class_percentages = (let_is_class_percentages * 100).round(2)

print("\nLET_IS class percentages:")
print(let_is_class_percentages)

# 
print("\nBinary target distributions:")

for column_name in y.columns:
    if column_name != "LET_IS":

        target_distribution = y[column_name].value_counts()

        target_distribution = target_distribution.sort_index()

        print(f"\n{column_name}:")
        print(target_distribution)

LET_IS class distribution:
LET_IS
0    1429
1     110
2      18
3      54
4      23
5      12
6      27
7      27
Name: count, dtype: int64

LET_IS class percentages:
LET_IS
0    84.06
1     6.47
2     1.06
3     3.18
4     1.35
5     0.71
6     1.59
7     1.59
Name: proportion, dtype: float64

Binary target distributions:

FIBR_PREDS:
FIBR_PREDS
0    1530
1     170
Name: count, dtype: int64

PREDS_TAH:
PREDS_TAH
0    1680
1      20
Name: count, dtype: int64

JELUD_TAH:
JELUD_TAH
0    1658
1      42
Name: count, dtype: int64

FIBR_JELUD:
FIBR_JELUD
0    1629
1      71
Name: count, dtype: int64

A_V_BLOK:
A_V_BLOK
0    1643
1      57
Name: count, dtype: int64

OTEK_LANC:
OTEK_LANC
0    1541
1     159
Name: count, dtype: int64

RAZRIV:
RAZRIV
0    1646
1      54
Name: count, dtype: int64

DRESSLER:
DRESSLER
0    1625
1      75
Name: count, dtype: int64

ZSN:
ZSN
0    1306
1     394
Name: count, dtype: int64

REC_IM:
REC_IM
0    1541
1     159
Name: count, dtype: int64

P_IM_STEN:
P_IM_ST

We can use Iterative stratification to make sure when we do our test/train/validation splits, we are trying to get atleast some postive examples from the target features with fewer psotive outcomes(trying to adress class imbalance from the split to start with). Because Iterative startification won't work with both binary target features and the multi-class output we can One Hot Code the LET_IS column first. Then once the splitting has been done, we can use the row indices defined in the split and apply it to X and y data sets to preserve LET_IS as a multiclass output so the target features will remain the same during traning.

## Do the Split - Iterative Stratification

In [70]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

In [72]:
#grab LET_IS column
let_is = y["LET_IS"]

#one hot encodeing 
let_is_encode = pd.get_dummies(let_is, prefix="LET_IS", dtype=int)

#Drop LET_IS from the targets
dropped_let_is = y.drop(columns=["LET_IS"])

#Combine the encoded LET_IS columns with the bianry ones
new_y = pd.concat([dropped_let_is, let_is_encode], axis=1)

print(new_y.shape)

(1700, 19)


In [73]:
#split the train set first
first_split = MultilabelStratifiedShuffleSplit(n_splits=1,test_size=0.30,random_state=42)

# do the split and create indices
train_split = first_split.split(X, new_y)

# Grab the indices
training_indices, to_split_indices = next(train_split)

print("Training indices:", len(training_indices))
print("Temporary indices:", len(to_split_indices))

Training indices: 1190
Temporary indices: 510


In [74]:
#split the to_split set into validation and test sets
second_split = MultilabelStratifiedShuffleSplit(n_splits=1,test_size=0.50,random_state=42)

#grab the rows corresponsing to the indices in to_split_set
to_split_X = X.iloc[to_split_indices]
to_split_y = new_y.iloc[to_split_indices]

#split and create indices
validation_test_split = second_split.split(to_split_X, to_split_y)

#grab the indices - these are relative to the to_split set so we have to map them to the actual dataset
validation_indices_1, test_indices_1 = next(validation_test_split)

#convert indices so they are relative to actual dataset
validation_indices = to_split_indices[validation_indices_1]
test_indices = to_split_indices[test_indices_1]


print("Training:", len(training_indices))
print("Validation:", len(validation_indices))
print("Test:", len(test_indices))



Training: 1190
Validation: 255
Test: 255


Now that we have the split indices we can grab those index rows from our original datset (X and y) without the one hot encoded LET_IS

In [75]:
#get X_train set
X_train_rows = X.iloc[training_indices]
X_train = X_train_rows.copy()

#get y_train set
y_train_rows = y.iloc[training_indices]
y_train = y_train_rows.copy()

#get X_validation set
X_validation_rows = X.iloc[validation_indices]
X_validation = X_validation_rows.copy()

#get y_validation set
y_validation_rows = y.iloc[validation_indices]
y_validation = y_validation_rows.copy()

#get X_test set
X_test_rows = X.iloc[test_indices]
X_test = X_test_rows.copy()

#get y_test set
y_test_rows = y.iloc[test_indices]
y_test = y_test_rows.copy()

print(X_train.shape)
print(y_train.shape)

print(X_validation.shape)
print(y_validation.shape)

print(X_test.shape)
print(y_test.shape)

(1190, 107)
(1190, 12)
(255, 107)
(255, 12)
(255, 107)
(255, 12)


Check distribution of targets in splits

In [76]:
# Store the three target datasets
target_datasets = {
    "Training": y_train,
    "Validation": y_validation,
    "Test": y_test
}

# Go through each target column
for target in y.columns:

    print("\n")
    print("Target:", target)
   

    # Go through each dataset
    for dataset_name, dataset in target_datasets.items():

        print("\n", dataset_name)

        # Get the target column
        target_values = dataset[target]

        # Calculate the number of occurrences of each class
        counts = target_values.value_counts()

        # Sort the counts by class
        counts = counts.sort_index()

        # Calculate the proportion of each class
        proportions = target_values.value_counts(normalize=True)

        # Sort the proportions by class
        proportions = proportions.sort_index()

        # Convert proportions into percentages
        percentages = proportions * 100

        # Create a table containing counts and percentages
        distribution = pd.DataFrame({
            "Count": counts,
            "Percentage": percentages
        })

        # Round percentages to 2 decimal places
        distribution = distribution.round(2)

        # Display the results
        print(distribution)



Target: FIBR_PREDS

 Training
            Count  Percentage
FIBR_PREDS                   
0            1071        90.0
1             119        10.0

 Validation
            Count  Percentage
FIBR_PREDS                   
0             229        89.8
1              26        10.2

 Test
            Count  Percentage
FIBR_PREDS                   
0             230        90.2
1              25         9.8


Target: PREDS_TAH

 Training
           Count  Percentage
PREDS_TAH                   
0           1176       98.82
1             14        1.18

 Validation
           Count  Percentage
PREDS_TAH                   
0            252       98.82
1              3        1.18

 Test
           Count  Percentage
PREDS_TAH                   
0            252       98.82
1              3        1.18


Target: JELUD_TAH

 Training
           Count  Percentage
JELUD_TAH                   
0           1161       97.56
1             29        2.44

 Validation
           Count  Percentage


## IMPUTE THE DATASETS

We first split to prevent data leakage. It is important that the data in test and validation is unseen so we cannot use that to compute our imputation values. Therefore we have to compute using our training values and then apply that to our validation and test set. Same idea with scaling - we calculate values use train set and then use those to transform the validation and test set. We can impute based on the column types we defined earlier

Starting with features that are numerical - we can impute using the median. The median was chosen because the medical data can be extreme or have outliers and the median is not influenced by outliers.

In [77]:
#Check missing values before imputation

# Check missing values in training set
print("Training missing values:", X_train.isnull().sum())

# Check missing values in validation set
print("Validation missing values:", X_validation.isnull().sum())

# Check missing values in test set
print("Test missing values:", X_test.isnull().sum())

Training missing values: AGE               4
SEX               0
INF_ANAM          4
STENOK_AN        79
FK_STENOK        56
                 ..
GEPAR_S_n        10
ASP_S_n          10
TIKL_S_n          9
TRENT_S_n         9
EMR_RES_BLOOD     0
Length: 107, dtype: int64
Validation missing values: AGE               3
SEX               0
INF_ANAM          0
STENOK_AN        13
FK_STENOK         9
                 ..
GEPAR_S_n         4
ASP_S_n           5
TIKL_S_n          4
TRENT_S_n         4
EMR_RES_BLOOD     0
Length: 107, dtype: int64
Test missing values: AGE               1
SEX               0
INF_ANAM          0
STENOK_AN        14
FK_STENOK         8
                 ..
GEPAR_S_n         3
ASP_S_n           2
TIKL_S_n          3
TRENT_S_n         3
EMR_RES_BLOOD     0
Length: 107, dtype: int64


In [78]:
from sklearn.impute import SimpleImputer

# Create the median imputer
median_imputer = SimpleImputer(strategy="median")

# Grab the numerical features from the training set
X_train_numerical = X_train[numerical_measurement_column_names_keep]

#calculate the median based on training data only
median_imputer.fit(X_train_numerical)

#impute the median values for the numerical columns in training set
X_train[numerical_measurement_column_names_keep] = median_imputer.transform(X_train[numerical_measurement_column_names_keep])

#use the same median and apply to validation and test set(once again we use the calculated values from traning)
X_validation[numerical_measurement_column_names_keep] = median_imputer.transform(X_validation[numerical_measurement_column_names_keep])
X_test[numerical_measurement_column_names_keep] = median_imputer.transform(X_test[numerical_measurement_column_names_keep])




Now we deal with non-numerical features - the categorical and binary columns. For these we will use mode as the most frequent value will be selected to fill in missing values.

In [79]:
#create the mode imputer
mode_imputer = SimpleImputer(strategy="most_frequent")

#Grab the categorical and binary features from training set
X_train_other = X_train[other_feature_column_names_keep]

#calculate the mode based on training data only
mode_imputer.fit(X_train_other)

#impute the mode values for the columns in training set
X_train[other_feature_column_names_keep] = mode_imputer.transform(X_train[other_feature_column_names_keep])

#use the same mode and apply to validation and test set(once again we use the calculated values from traning)
X_validation[other_feature_column_names_keep] = mode_imputer.transform(X_validation[other_feature_column_names_keep])
X_test[other_feature_column_names_keep] = mode_imputer.transform(X_test[other_feature_column_names_keep])

Check all missing values are gone in Training, Validation and Test sets

In [80]:
# Check missing values in training set
print("Training missing values:", X_train.isnull().sum())

# Check missing values in validation set
print("Validation missing values:", X_validation.isnull().sum())

# Check missing values in test set
print("Test missing values:", X_test.isnull().sum())

Training missing values: AGE              0
SEX              0
INF_ANAM         0
STENOK_AN        0
FK_STENOK        0
                ..
GEPAR_S_n        0
ASP_S_n          0
TIKL_S_n         0
TRENT_S_n        0
EMR_RES_BLOOD    0
Length: 107, dtype: int64
Validation missing values: AGE              0
SEX              0
INF_ANAM         0
STENOK_AN        0
FK_STENOK        0
                ..
GEPAR_S_n        0
ASP_S_n          0
TIKL_S_n         0
TRENT_S_n        0
EMR_RES_BLOOD    0
Length: 107, dtype: int64
Test missing values: AGE              0
SEX              0
INF_ANAM         0
STENOK_AN        0
FK_STENOK        0
                ..
GEPAR_S_n        0
ASP_S_n          0
TIKL_S_n         0
TRENT_S_n        0
EMR_RES_BLOOD    0
Length: 107, dtype: int64


We have now got 0 missing values in our feature set. The target set did not have any missing values so no need to remove those rows or deal with that.

## Scale Numerical Features

We only scale the numerical features so that they don't overpower the data - we want all features to be treated as "equal" importance towards predictions. We should have a mean of aorund 0 and std of 1.

In [81]:
# Check the mean of unscaled numerical features
print("Training means:")
means = X_train[numerical_measurement_column_names_keep].mean()
rounded_means = means.round(2)
print(rounded_means)

# Check the standard deviation of unscaled numerical features
print("\nTraining standard deviations:")
standard_deviations = X_train[numerical_measurement_column_names_keep].std()
rounded_standard_deviations = standard_deviations.round(2)
print(rounded_standard_deviations)

Training means:
K_BLOOD         4.17
NA_BLOOD      136.34
ALT_BLOOD       0.46
AST_BLOOD       0.26
L_BLOOD         8.76
ROE            13.20
AGE            61.95
S_AD_ORIT     133.23
D_AD_ORIT      81.98
S_AD_KBRIG    138.94
D_AD_KBRIG     80.34
dtype: float64

Training standard deviations:
K_BLOOD        0.68
NA_BLOOD       5.82
ALT_BLOOD      0.36
AST_BLOOD      0.19
L_BLOOD        3.27
ROE           11.08
AGE           11.25
S_AD_ORIT     28.72
D_AD_ORIT     16.62
S_AD_KBRIG    21.35
D_AD_KBRIG    11.80
dtype: float64


In [82]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

#detrmine how to scale using training data only
scaler.fit(X_train[numerical_measurement_column_names_keep])

#scale the training set
X_train[numerical_measurement_column_names_keep] = scaler.transform(X_train[numerical_measurement_column_names_keep])

#scale validation set
X_validation[numerical_measurement_column_names_keep] = scaler.transform(X_validation[numerical_measurement_column_names_keep])

#scale test set
X_test[numerical_measurement_column_names_keep] = scaler.transform(X_test[numerical_measurement_column_names_keep])


In [83]:
# Check the mean of scaled numerical features
print("Training means:")
means = X_train[numerical_measurement_column_names_keep].mean()
rounded_means = means.round(2)
print(rounded_means)

# Check the standard deviation of scaled numerical features
print("\nTraining standard deviations:")
standard_deviations = X_train[numerical_measurement_column_names_keep].std()
rounded_standard_deviations = standard_deviations.round(2)
print(rounded_standard_deviations)

Training means:
K_BLOOD      -0.0
NA_BLOOD      0.0
ALT_BLOOD     0.0
AST_BLOOD     0.0
L_BLOOD      -0.0
ROE           0.0
AGE          -0.0
S_AD_ORIT    -0.0
D_AD_ORIT     0.0
S_AD_KBRIG   -0.0
D_AD_KBRIG    0.0
dtype: float64

Training standard deviations:
K_BLOOD       1.0
NA_BLOOD      1.0
ALT_BLOOD     1.0
AST_BLOOD     1.0
L_BLOOD       1.0
ROE           1.0
AGE           1.0
S_AD_ORIT     1.0
D_AD_ORIT     1.0
S_AD_KBRIG    1.0
D_AD_KBRIG    1.0
dtype: float64


The nominal values have already been one hot encoded so they are now represented as binary columns. The ordinal features have ranking in which order or ranking matters and has to be preserved. We can use a min-max scaler to preserve order and have it be set between 0 and 1. This makes it comparable to the binary columns. Once again scale based on training set then, apply to test and validation. Going to also scale ZSN_A treating it as an ordinal now but later might treat split it by it's severity branches.

In [84]:
ordinal_column_names_keep = ordinal_column_names_keep + ["ZSN_A"]

# Check means before scaling
print("Means before scaling:")
print(X_train[ordinal_column_names_keep].mean().round(3))

# Check standard deviations before scaling
print("\nStandard deviations before scaling:")
print(X_train[ordinal_column_names_keep].std().round(3))

Means before scaling:
INF_ANAM      0.551
STENOK_AN     2.088
FK_STENOK     1.182
GB            1.381
DLIT_AG       2.842
ant_im        1.488
lat_im        0.850
inf_im        0.979
post_im       0.239
TIME_B_S      4.488
R_AB_1_n      0.313
R_AB_2_n      0.129
NOT_NA_1_n    0.332
IBS_POST      1.168
NA_R_1_n      0.490
NA_R_2_n      0.088
NOT_NA_2_n    0.107
ZSN_A         0.184
dtype: float64

Standard deviations before scaling:
INF_ANAM      0.840
STENOK_AN     2.435
FK_STENOK     1.028
GB            1.087
DLIT_AG       3.107
ant_im        1.699
lat_im        0.876
inf_im        1.380
post_im       0.668
TIME_B_S      2.863
R_AB_1_n      0.631
R_AB_2_n      0.407
NOT_NA_1_n    0.630
IBS_POST      0.817
NA_R_1_n      0.794
NA_R_2_n      0.343
NOT_NA_2_n    0.397
ZSN_A         0.635
dtype: float64


In [85]:
from sklearn.preprocessing import MinMaxScaler

mix_max_scaler = MinMaxScaler()

#detrmine how to scale using training data only
mix_max_scaler.fit(X_train[ordinal_column_names_keep])

# Scale training set
X_train[ordinal_column_names_keep] = mix_max_scaler.transform(X_train[ordinal_column_names_keep])

# Scale validation set using only the training calc values
X_validation[ordinal_column_names_keep] = mix_max_scaler.transform(X_validation[ordinal_column_names_keep])

# Scale test set using only training calc values
X_test[ordinal_column_names_keep] = mix_max_scaler.transform(X_test[ordinal_column_names_keep])

In [86]:
# Check means after scaling
print("Means after scaling:")
print(X_train[ordinal_column_names_keep].mean().round(3))

# Check standard deviations after scaling
print("\nStandard deviations after scaling:")
print(X_train[ordinal_column_names_keep].std().round(3))

# Check minimum values
print("\nMinimum values:")
print(X_train[ordinal_column_names_keep].min())

# Check maximum values
print("\nMaximum values:")
print(X_train[ordinal_column_names_keep].max())

Means after scaling:
INF_ANAM      0.184
STENOK_AN     0.348
FK_STENOK     0.296
GB            0.460
DLIT_AG       0.406
ant_im        0.372
lat_im        0.212
inf_im        0.245
post_im       0.060
TIME_B_S      0.436
R_AB_1_n      0.104
R_AB_2_n      0.064
NOT_NA_1_n    0.083
IBS_POST      0.584
NA_R_1_n      0.122
NA_R_2_n      0.044
NOT_NA_2_n    0.036
ZSN_A         0.046
dtype: float64

Standard deviations after scaling:
INF_ANAM      0.280
STENOK_AN     0.406
FK_STENOK     0.257
GB            0.362
DLIT_AG       0.444
ant_im        0.425
lat_im        0.219
inf_im        0.345
post_im       0.167
TIME_B_S      0.358
R_AB_1_n      0.210
R_AB_2_n      0.204
NOT_NA_1_n    0.157
IBS_POST      0.409
NA_R_1_n      0.198
NA_R_2_n      0.171
NOT_NA_2_n    0.132
ZSN_A         0.159
dtype: float64

Minimum values:
INF_ANAM      0.0
STENOK_AN     0.0
FK_STENOK     0.0
GB            0.0
DLIT_AG       0.0
ant_im        0.0
lat_im        0.0
inf_im        0.0
post_im       0.0
TIME_B_S      

## FINAL PREPROCESSING NOTES FOR REPORT

We have analyzed the data and dropped columns with information available only after 48h of hospitalization: columns 95(R_AB_3_n), 102(NA_R_3_n), and 105(NOT_NA_3_n). We have then dropped additonal columns with a high level of missingness: KFK_BLOOD has only 4 values and IBS_NASL has 72 in the entire dataset. This is 99.764706% and 95.764706% of data missing for ecah column respectively. Columns missing above 50% of the data were inspected and analyzed further: D_AD_KBRIG and S_AD_KBRIG - both over 63.29% missing data. These are columns signifiying blood presure measurments taken by emergency repsonse team. The high level of missing data makes sense - not everyone might have arrived to the hospital in an ambulance. A correlation analysis was run between these features and D_AD_ORIT and S_AD_ORIT. The correlation between D_AD_KBRIG and D_AD_ORIT was 0.5559600310828933 while the correlation between S_AD_KBRIG vs S_AD_ORIT was 0.6113647279368555 - both of which are moderately strong postive correlation scores. However they do not justify dropping columns D_AD_KBRIG and S_AD_KBRIG - therefore we created a binary indicator column called: EMR_RES_BLOOD. Only one indicator column was created as both D_AD_KBRIG and S_AD_KBRIG were missing at the same time - we encoded 0 for value missing and 1 more value present. Then these columns were kept along with the indicator column and the D_AD_KBRIG and S_AD_KBRIG were imputed as normal numerical feature columns(later on). ## --- talk about figuring out and verisfy different feature column types and putting them in lists - the partially order4d one keep in mind for later but treated normally for now. Then iterative stratification to try and get target postive cases in ecah split(test, validate, train). And then impute numeric columns with median and categorical with mode(including the partially ordered one). Then scaled values(inlcuded partially scaled one - ZSN_A) for now - in future might split by severity branch.